In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import sys

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully logged into Hugging Face!")
except Exception as e:
    print("ERROR: Could not find HF_TOKEN.")
    sys.exit("Stopping execution.")

import torch
import os
from PIL import Image
from tqdm import tqdm

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class_mapping = {
    "Bacterial Leaf Blight": 0, "Bacterial Streak": 1, "Bakanae": 2, 
    "Brown Spot": 3, "False Smut": 4, "Grassy Stunt Virus": 5, 
    "Healthy Leaf": 6, "Hispa": 7, "Leaf Blast": 8, "Leaf Scald": 9, 
    "Leaf Smut": 10, "Narrow Brown Spot": 11, "Neck Blast": 12, 
    "Ragged Stunt Virus": 13, "Sheath Blight": 14, "Sheath Rot": 15, 
    "Stem Rot": 16, "Tungro": 17, "Insect Affected": 18,
}

class_names = list(class_mapping.keys())
options_string = ", ".join(class_names)

class PlantDocDatasetVLM:
    def __init__(self, root):
        self.samples = []
        
        for folder_name in os.listdir(root):
            mapped_label = None
            
            if folder_name in class_mapping:
                mapped_label = class_mapping[folder_name]
            else:
                clean_folder = folder_name.lower().replace('_valid', '').replace('_test', '').replace('_', ' ')
                for key, val in class_mapping.items():
                    clean_key = key.lower().replace('_', ' ')
                    if clean_key == clean_folder:
                        mapped_label = val
                        break

            if mapped_label is not None:
                class_path = os.path.join(root, folder_name)
                for img in os.listdir(class_path):
                    if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append(
                            (os.path.join(class_path, img), mapped_label)
                        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        return image, label

plantdoc_test_root = "/kaggle/input/datasets/vishnuawasthi/rice-test-data/Rice Disease Dataset Test"
test_dataset = PlantDocDatasetVLM(plantdoc_test_root)
print(f"Rice dataset Test size: {len(test_dataset)}")

model_id = "google/gemma-3-4b-it" 

print(f"\nLoading {model_id} in 4-bit precision...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)
model.eval()
print("Gemma 3 loaded successfully.\n")

all_preds = []
all_labels = []

print("Starting Zero-Shot Vision-Language Evaluation...")

with torch.no_grad():
    for i in tqdm(range(len(test_dataset))):
        raw_image, label_idx = test_dataset[i]
        
        prompt = f"""You are an expert agricultural AI. Analyze this image of a rice plant.
Identify the disease from this exact list of options: [{options_string}].
Answer ONLY with the exact name of the disease from the list. Do not include any other text."""

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        text_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        
        inputs = processor(
            text=text_prompt, 
            images=raw_image, 
            return_tensors="pt"
        ).to(device)
        
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=15, 
            temperature=0.0,  
            do_sample=False
        )
        
        input_length = inputs["input_ids"].shape[1]
        generated_text = processor.decode(generated_ids[0][input_length:], skip_special_tokens=True).strip()
        
        predicted_idx = -1
        for idx, class_name in enumerate(class_names):
            if class_name.lower() in generated_text.lower():
                predicted_idx = idx
                break
                
        if predicted_idx == -1:
            predicted_idx = 0 
            
        all_preds.append(predicted_idx)
        all_labels.append(label_idx)

test_acc = accuracy_score(all_labels, all_preds)

print("\n==============================")
print(f"GEMMA 3 ZERO-SHOT TEST ACCURACY: {test_acc:.4f}")
print("==============================")

cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:")
print(cm)

print("\nPer Class Accuracy:")
for i, class_name in enumerate(class_names):
    class_total = cm[i].sum()
    if class_total > 0:
        class_correct = cm[i][i]
        acc = class_correct / class_total
        print(f"{class_name}: {acc:.4f}")
    else:
        print(f"{class_name}: N/A (0 samples)")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

Successfully logged into Hugging Face!
Using device: cuda
Rice dataset Test size: 465

Loading google/gemma-3-4b-it in 4-bit precision...


The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma 3 loaded successfully.

Starting Zero-Shot Vision-Language Evaluation...


100%|██████████| 465/465 [22:26<00:00,  2.89s/it]


GEMMA 3 ZERO-SHOT TEST ACCURACY: 0.0645

Confusion Matrix:
[[ 0  3  1 20  0  0  0  0  5  1  0  0  0  0  0  0  0  0  0]
 [ 5  3  2  3  0  0  0  0  2  0  0  0  0  0  0  0  0  0  0]
 [ 0  3  4  0  0  0  0  0  7  0  0  0  0  0  0  0  0  1  0]
 [ 3  4  0 20  0  0  0  0  1  2  0  0  0  0  0  0  0  0  0]
 [ 1  0 14  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 1  0  0  9  0  0  0  0  5  0  0  0  0  0  0  0  0  0  0]
 [ 2  4  0  7  0  0  1  0 15  0  0  0  0  0  0  0  0  0  1]
 [ 5 11  0 10  0  0  0  0  4  0  0  0  0  0  0  0  0  0  0]
 [ 2  4  0 23  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0]
 [ 4  2  0 17  0  0  0  0  6  1  0  0  0  0  0  0  0  0  0]
 [ 1  2  0 24  0  0  0  0  1  2  0  0  0  0  0  0  0  0  0]
 [ 2  5  0 19  0  0  0  0  4  0  0  0  0  0  0  0  0  0  0]
 [12  1  5  4  0  0  0  0  8  0  0  0  0  0  0  0  0  0  0]
 [ 1  0  1  2  0  0  0  0 11  0  0  0  0  0  0  0  0  0  0]
 [ 2  3  3 16  0  0  0  0  5  1  0  0  0  0  0  0  0  0  0]
 [ 0  5  2  8  0  0  0  0  0  0  0  0  0